# Full Extraction Pipeline Demo

End-to-end walk-through of the WUMaCat parameter extraction pipeline:

1. Pick a random seed paper from WUMaCat
2. Run ADS similarity search — keep papers with score > 100
3. Download PDFs
4. Classify each PDF as text or image
5. Classify relevance through all 3 stages (OCR image PDFs first so all papers go through the full pipeline)
6. Delete irrelevant PDFs
7. Extract parameters from text PDFs → flat catalogue
8. Extract parameters from image PDFs → append to catalogue

Each paper's extraction result is also saved as a JSON file under `data/extracted/`.

## Setup

In [ ]:
import os, sys, json, random, time, shutil
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv

# ── Project root & config ─────────────────────────────────────────────────────
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

with open(PROJECT_ROOT / "config.yaml") as fh:
    CFG = yaml.safe_load(fh)

def cfg_path(key: str) -> Path:
    """Resolve a config path relative to project root."""
    return PROJECT_ROOT / CFG["paths"][key]

# Ensure output directories exist
cfg_path("pdf_dir").mkdir(parents=True, exist_ok=True)
cfg_path("extracted_json_dir").mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("PDF dir      :", cfg_path("pdf_dir"))
print("Extracted dir:", cfg_path("extracted_json_dir"))

## Step 1 — Pick a random seed paper from WUMaCat

In [ ]:
wumacat = pd.read_csv(cfg_path("wumacat_csv"))
print(f"WUMaCat: {len(wumacat)} systems, {wumacat['Bibcode'].nunique()} unique bibcodes")

unique_bibcodes = wumacat["Bibcode"].dropna().unique().tolist()

random.seed(42)                          # fix seed for reproducibility
SEED_BIBCODE = random.choice(unique_bibcodes)

seed_papers = wumacat[wumacat["Bibcode"] == SEED_BIBCODE]
print(f"\nSeed bibcode : {SEED_BIBCODE}")
print(f"Objects in it: {len(seed_papers)}")
seed_papers[["Name", "Bibcode", "Type", "P", "q", "T1", "T2"]].head()

## Step 2 — ADS similarity search (score > 100)

In [ ]:
from src.ads_parser import find_similar_papers

MIN_SCORE = CFG["extraction"]["similarity_score_threshold"]
MAX_RESULTS = CFG["extraction"]["max_similar_results"]

result = find_similar_papers(
    bibcode=SEED_BIBCODE,
    max_results=MAX_RESULTS,
    fields=["bibcode", "title", "year", "pub", "score"],
    min_score=MIN_SCORE,
)

if result is None or not result["papers"]:
    raise RuntimeError("Similarity search returned no results.")

similar_papers = result["papers"]
print(f"\nTotal found   : {result['total_found']}")
print(f"Retrieved     : {result['retrieved']}")
print(f"Score >= {MIN_SCORE} : {result['score_filtered']}")

sim_df = pd.DataFrame(similar_papers)
sim_df["title"] = sim_df["title"].apply(lambda t: t[0] if isinstance(t, list) else t)
sim_df = sim_df.sort_values("score", ascending=False).reset_index(drop=True)
sim_df[["bibcode", "year", "score", "title"]].head(20)

## Step 3 — Download PDFs

In [ ]:
from src.ads_parser import download_pdfs

candidate_bibcodes = sim_df["bibcode"].tolist()
pdf_dir = cfg_path("pdf_dir")

# ── Pre-check: which PDFs are already on disk? ────────────────────────────
already_have, need_download = [], []
for bib in candidate_bibcodes:
    local = pdf_dir / f"{bib}.pdf"
    if local.exists():
        already_have.append(bib)
    else:
        need_download.append(bib)

print(f"Candidates    : {len(candidate_bibcodes)}")
print(f"Already local : {len(already_have)}")
print(f"To download   : {len(need_download)}")
if already_have:
    print("\nSkipping (already downloaded):")
    for b in already_have:
        print(f"  {b}")

# ── Download only what is missing ─────────────────────────────────────────
dl = download_pdfs(
    bibcodes=need_download,
    output_dir=str(pdf_dir),
    delay_between_requests=CFG["pipeline"]["delay_between_requests"],
    skip_existing=False,   # already filtered above
) if need_download else {"pdf_files": {}, "no_source": [], "failed": []}

print(f"\nFreshly downloaded: {len(dl['pdf_files'])}")
print(f"No source         : {len(dl['no_source'])}")
print(f"Failed            : {len(dl['failed'])}")

# ── Merge: already-on-disk + freshly downloaded (both now have full paths) ─
available = {bib: str(pdf_dir / f"{bib}.pdf") for bib in already_have}
available.update({bib: path for bib, path in dl["pdf_files"].items() if path})

print(f"\nAvailable PDFs: {len(available)}")

## Step 4 — Classify PDFs: text vs image

In [ ]:
from src.pdf_utils import detect_pdf_type

pdf_types = {}   # bibcode -> "text" | "image"

for bib, pdf_path in available.items():
    info = detect_pdf_type(pdf_path)
    pdf_types[bib] = info.pdf_type

text_bibs  = [b for b, t in pdf_types.items() if t == "text"]
image_bibs = [b for b, t in pdf_types.items() if t == "image"]

print(f"Text PDFs : {len(text_bibs)}")
print(f"Image PDFs: {len(image_bibs)}")

type_df = pd.DataFrame(
    [(b, available[b], pdf_types[b]) for b in available],
    columns=["bibcode", "pdf_path", "pdf_type"],
).sort_values("pdf_type")
type_df

## Step 5 — Relevance classification through all 3 stages

**Stage 1** — keyword scoring (score ≥ 2.0 + system_type hit → relevant; < 0.5 → not_relevant)  
**Stage 2** — abstract cosine similarity to WUMaCat centroid (≥ 0.840 → relevant; < 0.836 → not_relevant)  
**Stage 3** — conclusions cosine similarity (≥ 0.820 → relevant; otherwise not_relevant)

Image PDFs have no embedded text, so they are OCR'd with EasyOCR first.  
The OCR output is cached and reused in Step 8 to avoid processing each PDF twice.

In [ ]:
from src.classifier import classify_paper
from src.embeddings import load_centroid
from src.pdf_utils import ocr_pdf
from src.param_extractor import _normalize

centroid = load_centroid()

clf_results = {}   # bibcode -> ClassificationResult
ocr_cache   = {}   # bibcode -> normalised OCR text (image PDFs only)

STAGE_ICONS = {"relevant": "OK", "not_relevant": "NO", "borderline": "--", "uncertain": "??"}

for bib, pdf_path in available.items():
    print(f"\n{'─'*60}")
    print(f"  {bib}  [{pdf_types[bib]}]")

    ocr_text = None
    if pdf_types[bib] == "image":
        print("  OCR (easyocr) ...", end=" ", flush=True)
        raw = ocr_pdf(pdf_path, engine="easyocr")
        ocr_text = _normalize(raw)
        ocr_cache[bib] = ocr_text
        print(f"done ({len(ocr_text):,} chars)")

    clf = classify_paper(pdf_path, centroid=centroid, text=ocr_text)
    clf_results[bib] = clf

    # Print per-stage breakdown
    for s in clf.stages:
        icon = STAGE_ICONS.get(s.verdict, "?")
        note = f"  ({s.note})" if s.note else ""
        print(f"    Stage {s.stage}: score={s.score:.4f}  [{icon}] {s.verdict}{note}")

    final_icon = {"relevant": "RELEVANT", "not_relevant": "NOT RELEVANT",
                  "uncertain": "UNCERTAIN"}.get(clf.verdict, clf.verdict)
    print(f"  => {final_icon}  (exit at stage {clf.exit_stage})")

print(f"\n{'─'*60}")
relevant_bibs     = [b for b, c in clf_results.items() if c.verdict == "relevant"]
not_relevant_bibs = [b for b, c in clf_results.items() if c.verdict == "not_relevant"]
uncertain_bibs    = [b for b, c in clf_results.items() if c.verdict == "uncertain"]

print(f"Relevant    : {len(relevant_bibs)}")
print(f"Uncertain   : {len(uncertain_bibs)}")
print(f"Not relevant: {len(not_relevant_bibs)}")

## Step 6 — Delete irrelevant PDFs

In [ ]:
deleted = []
for bib in not_relevant_bibs:
    pdf_path = available[bib]
    if pdf_path and Path(pdf_path).exists():
        os.remove(pdf_path)
        deleted.append(pdf_path)
        print(f"  Deleted: {Path(pdf_path).name}")

print(f"\nDeleted {len(deleted)} irrelevant PDFs.")

# Uncertain papers are kept — treat as relevant for extraction
to_process = relevant_bibs + uncertain_bibs
process_text  = [b for b in to_process if pdf_types.get(b) == "text"]
process_image = [b for b in to_process if pdf_types.get(b) == "image"]

print(f"\nTo process — text : {len(process_text)}")
print(f"To process — image: {len(process_image)}")

## Helper — save extraction result as JSON

In [ ]:
def save_result_json(result, out_dir: Path) -> Path:
    """
    Serialise a PaperResult to JSON and write it to out_dir/{bibcode}.json.
    Returns the output path.
    """
    payload = {
        "bibcode":    result.bibcode,
        "pdf_path":   result.pdf_path,
        "verdict":    result.verdict,
        "exit_stage": result.exit_stage,
        "error":      result.error,
        "objects":    [],
    }
    for row in result.rows:
        d = row.to_dict()
        # Convert NaN to None for valid JSON
        obj = {k: (None if isinstance(v, float) and v != v else v) for k, v in d.items()}
        payload["objects"].append(obj)

    out_path = out_dir / f"{result.bibcode}.json"
    with open(out_path, "w") as fh:
        json.dump(payload, fh, indent=2)
    return out_path

## Step 7 — Extract parameters from text PDFs

In [ ]:
from src.pipeline import process_paper
from src.catalogue import ObjectCatalogue

cat = ObjectCatalogue()
json_dir = cfg_path("extracted_json_dir")

print(f"Processing {len(process_text)} text PDF(s) ...\n")

for bib in process_text:
    pdf_path = available[bib]
    print(f"  [{bib}]")
    try:
        result = process_paper(
            pdf_path,
            centroid=centroid,
            skip_classification=True,   # already classified in Step 5
            verbose=False,
        )
        cat.add_paper_result(result)
        json_out = save_result_json(result, json_dir)
        print(f"    Objects extracted : {len(result.rows)}")
        print(f"    JSON saved        : {json_out.name}")
    except Exception as exc:
        print(f"    ERROR: {exc}")

df_text = cat.to_dataframe()
print(f"\nCatalogue after text PDFs: {len(df_text)} rows")
df_text[["Name", "bibcode", "q", "q_src", "T1", "T2", "T2_T1", "T2_T1_src"]].head(10)

## Step 8 — Extract parameters from image PDFs, append to catalogue

OCR text computed in Step 5 is reused here — no second OCR pass.

In [ ]:
print(f"Processing {len(process_image)} image PDF(s) ...\n")

for bib in process_image:
    pdf_path = available[bib]
    print(f"  [{bib}]")
    try:
        # Pass the cached OCR text — pipeline skips re-OCR when text is supplied
        result = process_paper(
            pdf_path,
            centroid=centroid,
            skip_classification=True,
            verbose=False,
            ocr_text=ocr_cache.get(bib),   # reuse OCR output from Step 5
        )
        cat.add_paper_result(result)
        json_out = save_result_json(result, json_dir)
        print(f"    Objects extracted : {len(result.rows)}")
        print(f"    JSON saved        : {json_out.name}")
    except Exception as exc:
        print(f"    ERROR: {exc}")

df_all = cat.to_dataframe()
print(f"\nCatalogue after image PDFs: {len(df_all)} rows  (was {len(df_text)})")

## Final catalogue

In [ ]:
out_csv = cfg_path("catalogue_csv")
cat.save(str(out_csv))

cat.summary()
cat.print_conflicts()

print("\nExtracted JSON files:")
for f in sorted(json_dir.glob("*.json")):
    print(f"  {f.name}")

In [ ]:
# Preview key columns of the final table
key_cols = ["Name", "bibcode", "P", "P_src", "q", "q_src",
            "i", "T1", "T2", "T2_T1", "T2_T1_src",
            "f", "r1p", "r2p", "Type", "Solver"]
available_cols = [c for c in key_cols if c in df_all.columns]

df_all[available_cols].to_string(index=False, na_rep="—")